In [ ]:
import pandas as pd

# ==============================
# FILE PATHS
# ==============================
mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

# ==============================
# READ FILES
# ==============================
mapping = pd.read_excel(mapping_file)
comparison = pd.read_excel(comparison_file)

mapping.columns = mapping.columns.str.strip()
comparison.columns = comparison.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

# Clean keys
mapping['Sub_Label'] = normalize(mapping['Sub_Label'])
mapping['Main_Label'] = normalize(mapping['Main_Label'])
comparison['Material'] = normalize(comparison['Material'])

mapping['Sub_Count'] = pd.to_numeric(mapping['Sub_Count'], errors='coerce').fillna(0)

# Merge
merged = mapping.merge(
    comparison,
    left_on='Sub_Label',
    right_on='Material',
    how='left'
)

# Identify columns
indent_cols = sorted([c for c in merged.columns if 'Indent' in c])
actual_cols = sorted([c for c in merged.columns if 'Production Plan' in c])

# Prepare result
child_parts = merged['Main_Label'].unique()
wide_df = pd.DataFrame({'Child_Part': child_parts})

# ==============================
# LOOP THROUGH DAYS
# ==============================
for indent_col, actual_col in zip(indent_cols, actual_cols):

    date = indent_col.split()[0]

    merged['Tentative'] = pd.to_numeric(merged[indent_col], errors='coerce').fillna(0)
    merged['Actual'] = pd.to_numeric(merged[actual_col], errors='coerce').fillna(0)

    merged['Tentative_Child'] = merged['Tentative'] * merged['Sub_Count']
    merged['Actual_Child'] = merged['Actual'] * merged['Sub_Count']

    daily = merged.groupby('Main_Label').agg({
        'Actual_Child': 'sum',
        'Tentative_Child': 'sum'
    }).reset_index()

    daily.rename(columns={
        'Main_Label': 'Child_Part',
        'Actual_Child': f"{date} Actual",
        'Tentative_Child': f"{date} Tentative"
    }, inplace=True)

    wide_df = wide_df.merge(daily, on='Child_Part', how='left')

# Fill blanks
wide_df = wide_df.fillna(0)

print(wide_df.head())

# ==============================
# SAVE
# ==============================
output_file = "Child_Comparison_Wide.xlsx"
wide_df.to_excel(output_file, index=False)

print("\nSaved to:", output_file)


In [ ]:
import pandas as pd

mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

mapping = pd.read_excel(mapping_file)
comparison = pd.read_excel(comparison_file)

mapping.columns = mapping.columns.str.strip()
comparison.columns = comparison.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

mapping['Sub_Label'] = normalize(mapping['Sub_Label'])
mapping['Main_Label'] = normalize(mapping['Main_Label'])
comparison['Material'] = normalize(comparison['Material'])

mapping['Sub_Count'] = pd.to_numeric(mapping['Sub_Count'], errors='coerce').fillna(0)

merged = mapping.merge(
    comparison,
    left_on='Sub_Label',
    right_on='Material',
    how='left'
)

indent_cols = sorted([c for c in merged.columns if c.endswith('Indent')])
actual_cols = sorted([c for c in merged.columns if c.endswith('Production Plan')])

child_list = merged['Main_Label'].unique()
wide_df = pd.DataFrame({'Child_Part': child_list})

for indent_col, actual_col in zip(indent_cols, actual_cols):

    date = indent_col.split()[0]

    merged['Tentative'] = pd.to_numeric(merged[indent_col], errors='coerce').fillna(0)
    merged['Actual'] = pd.to_numeric(merged[actual_col], errors='coerce').fillna(0)

    merged['Tentative_Child'] = merged['Tentative'] * merged['Sub_Count']
    merged['Actual_Child'] = merged['Actual'] * merged['Sub_Count']

    daily = merged.groupby('Main_Label').agg({
        'Tentative_Child': 'sum',
        'Actual_Child': 'sum'
    }).reset_index()

    daily.rename(columns={
        'Main_Label': 'Child_Part',
        'Tentative_Child': f"{date} Tentative",
        'Actual_Child': f"{date} Actual"
    }, inplace=True)

    wide_df = wide_df.merge(daily, on='Child_Part', how='left')

wide_df = wide_df.fillna(0)

print(wide_df.head())

wide_df.to_excel("Child_Daily_Comparison_Wide.xlsx", index=False)

print("\nSaved: Child_Daily_Comparison_Wide.xlsx")


In [ ]:
import pandas as pd

bom_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

bom_df = pd.read_excel(bom_file)
comp_df = pd.read_excel(comparison_file)

bom_df.columns = bom_df.columns.str.strip()
comp_df.columns = comp_df.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

# Normalize keys
bom_df['Sub_Label'] = normalize(bom_df['Sub_Label'])
bom_df['Main_Label'] = normalize(bom_df['Main_Label'])
comp_df['Material'] = normalize(comp_df['Material'])

# Identify columns
indent_cols = sorted([c for c in comp_df.columns if c.endswith('Indent')])
actual_cols = sorted([c for c in comp_df.columns if c.endswith('Production Plan')])

# Prepare output
child_list = bom_df['Main_Label'].unique()
wide_df = pd.DataFrame({'Child_Part': child_list})

for indent_col, actual_col in zip(indent_cols, actual_cols):

    date = indent_col.split()[0]

    # Build lookup for this day
    comp_df['Tentative'] = pd.to_numeric(comp_df[indent_col], errors='coerce').fillna(0)
    comp_df['Actual'] = pd.to_numeric(comp_df[actual_col], errors='coerce').fillna(0)

    tent_lookup = dict(zip(comp_df['Material'], comp_df['Tentative']))
    act_lookup = dict(zip(comp_df['Material'], comp_df['Actual']))

    # Sequential logic
    results_tent = []
    results_act = []

    current_child = None
    running_tent = 0
    running_act = 0

    for _, row in bom_df.iterrows():

        child = row['Main_Label']
        switch = row['Sub_Label']
        usage = pd.to_numeric(row['Sub_Count'], errors='coerce') or 0

        tent_switch = tent_lookup.get(switch, 0)
        act_switch = act_lookup.get(switch, 0)

        tent_contrib = tent_switch * usage
        act_contrib = act_switch * usage

        if current_child is None:
            current_child = child

        if child != current_child:
            results_tent.append((current_child, running_tent))
            results_act.append((current_child, running_act))
            current_child = child
            running_tent = 0
            running_act = 0

        running_tent += tent_contrib
        running_act += act_contrib

    # Append last child
    if current_child is not None:
        results_tent.append((current_child, running_tent))
        results_act.append((current_child, running_act))

    tent_df = pd.DataFrame(results_tent, columns=['Child_Part', f"{date} Tentative"])
    act_df = pd.DataFrame(results_act, columns=['Child_Part', f"{date} Actual"])

    daily_df = tent_df.merge(act_df, on='Child_Part', how='outer')

    wide_df = wide_df.merge(daily_df, on='Child_Part', how='left')

wide_df = wide_df.fillna(0)

print(wide_df.head())

wide_df.to_excel("Child_Daily_Comparison_Sequential.xlsx", index=False)

print("\nSaved: Child_Daily_Comparison_Sequential.xlsx")
